# Tutorial 2: Data Visualizations and Data Summaries

**Course:** BCS1520 Statistics  
**Accompanies:** Lectures 1–2 (Data summaries, visualization, inductive inference)

---

## Why this tutorial matters

In Lectures 1 and 2 you learned that a handful of numbers — the mean, median, standard deviation, percentiles — can summarise thousands of observations. You also saw that the *right* visualization can reveal patterns that no single number captures: skewness, clusters, outliers, and differences between groups.

But how do you actually *compute* those summaries and *create* those visualizations when you're sitting in front of a real dataset? That is what this tutorial is for.

### The scenario

Imagine you are a software engineer responsible for the performance of a web application. Your application exposes several API endpoints (`/api/users`, `/api/products`, `/api/search`, etc.) across multiple servers. Every HTTP request is logged with its response time, status code, which server handled it, and whether the user was on a free or premium plan.

Your team wants answers to questions like:

- **How fast is our app?** What is a "typical" response time, and how much does it vary?
- **Which endpoints are slowest?** Are some API routes consistently worse than others?
- **Are there outliers?** Do some requests take absurdly long, and how many?
- **Does the server matter?** Is one machine dragging down performance?
- **Free vs. premium users** — is there a noticeable difference in the service they receive?

Each of these questions maps directly onto a statistical concept from the lectures. By the end of this tutorial you will be able to answer all of them with Python.


## Setup

We start by importing the libraries we will use throughout the tutorial and loading the dataset.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set a clean visual style for all plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Load the app performance dataset
df = pd.read_csv('app_performance.csv')
print(f"Dataset shape: {df.shape[0]} requests, {df.shape[1]} columns")
df.head()

Take a moment to look at the columns. Each row is one HTTP request. The columns record:

| Column | Meaning |
|--------|---------|
| `request_id` | Unique identifier for the request |
| `timestamp` | When the request was made |
| `endpoint` | Which API route was called |
| `method` | HTTP method (GET or POST) |
| `response_time_ms` | How long the server took to respond, in milliseconds |
| `status_code` | HTTP status (200 = OK, 404 = not found, 500 = server error, etc.) |
| `server` | Which backend server handled the request |
| `user_type` | Whether the user is on the free or premium plan |
| `payload_size_kb` | Size of the response payload in kilobytes |

The column we will focus on most is `response_time_ms` — the key performance metric.


---
# Section 1: Data Summaries in Depth

In lecture you learned about **measures of central tendency** (mean, median) and **measures of spread** (standard deviation, IQR, percentiles). Let's compute each of these for our response times and think about what they tell us.

### 1.1 Mean, Median, and Mode

Recall from lecture:
- The **mean** is the arithmetic average — it uses every data point, so it is sensitive to extreme values.
- The **median** is the middle value when the data is sorted — it is robust to outliers.
- The **mode** is the most frequent value — useful for categorical data but less informative for continuous measurements.

If the mean is much larger than the median, the distribution is **right-skewed** (pulled by a few very large values). For response times, this is common: most requests are fast, but some take very long.


In [ ]:
response_times = df['response_time_ms']

mean_rt = response_times.mean()
median_rt = response_times.median()

print(f"Mean response time:   {mean_rt:.2f} ms")
print(f"Median response time: {median_rt:.2f} ms")
print(f"Difference:           {mean_rt - median_rt:.2f} ms")

**What to notice:** Is the mean larger than the median? If so, by how much? This tells you the distribution is skewed to the right — a few slow requests are pulling the average up. In a performance context, this means the "average" response time may not represent the typical user experience very well. The median is often a better summary of what a *typical* request looks like.

### 1.2 Percentiles

Percentiles give you a richer picture than just the mean and median. The *p*-th percentile is the value below which *p*% of the data falls. In the software industry, the **p95** and **p99** response times are critical metrics: they tell you how bad things get for the slowest 5% or 1% of requests.

For example, an SLA (Service Level Agreement) might say: "99% of requests must complete in under 1000 ms."


In [ ]:
percentiles = [25, 50, 75, 90, 95, 99]
p_values = np.percentile(response_times, percentiles)

print("Percentile breakdown of response times:")
print("-" * 35)
for p, val in zip(percentiles, p_values):
    print(f"  p{p:<2d}:  {val:>8.2f} ms")

print(f"\nInterpretation: 95% of requests finish within {p_values[4]:.0f} ms.")
print(f"The slowest 1% take longer than {p_values[5]:.0f} ms.")

**What to notice:** Look at the jump between p90, p95, and p99. Is the gap between p95 and p99 much larger than the gap between p50 and p75? This "long tail" is characteristic of response time data — and it's exactly why percentiles matter more than the mean when monitoring real systems.

### 1.3 Standard Deviation and Variance

The **standard deviation** (σ) tells you how much individual values typically deviate from the mean. A small standard deviation means the data is tightly clustered; a large one means it is widely spread.

Recall from lecture: if the data is roughly normally distributed, about 68% of values fall within mean ± 1σ, and about 95% within mean ± 2σ. Let's check whether that holds for our response times.


In [ ]:
std_dev = response_times.std()
variance = response_times.var()

print(f"Standard deviation: {std_dev:.2f} ms")
print(f"Variance:           {variance:.2f} ms²")
print()

# Check the 68-95 rule
within_1sd = ((response_times >= mean_rt - std_dev) & (response_times <= mean_rt + std_dev)).mean()
within_2sd = ((response_times >= mean_rt - 2*std_dev) & (response_times <= mean_rt + 2*std_dev)).mean()

print(f"Mean ± 1σ: [{mean_rt - std_dev:.0f}, {mean_rt + std_dev:.0f}] ms")
print(f"  Actual % within this range: {100*within_1sd:.1f}% (expect ~68% if normal)")
print(f"Mean ± 2σ: [{mean_rt - 2*std_dev:.0f}, {mean_rt + 2*std_dev:.0f}] ms")
print(f"  Actual % within this range: {100*within_2sd:.1f}% (expect ~95% if normal)")

**What to notice:** If the percentages deviate a lot from 68% and 95%, the data is not well-described by a normal distribution. Response time data is typically right-skewed, so you may find that *more* than 68% falls within 1σ of the mean (because the right tail inflates σ beyond what the bulk of the data warrants).

### 1.4 IQR and Outlier Detection

The **Interquartile Range (IQR)** is Q3 − Q1, covering the middle 50% of the data. It is a robust measure of spread because it ignores the tails entirely.

The standard rule of thumb for **outliers** (which you may have seen in lecture): any value below Q1 − 1.5 × IQR or above Q3 + 1.5 × IQR is considered an outlier. In a performance context, outliers are the requests that took unusually long — worth investigating.


In [ ]:
Q1 = response_times.quantile(0.25)
Q3 = response_times.quantile(0.75)
IQR = Q3 - Q1

lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR
outliers = response_times[(response_times < lower_fence) | (response_times > upper_fence)]

print(f"Q1 (25th percentile): {Q1:.2f} ms")
print(f"Q3 (75th percentile): {Q3:.2f} ms")
print(f"IQR:                  {IQR:.2f} ms")
print(f"Lower fence:          {lower_fence:.2f} ms")
print(f"Upper fence:          {upper_fence:.2f} ms")
print(f"\nOutliers detected: {len(outliers)} out of {len(response_times)} requests ({100*len(outliers)/len(response_times):.1f}%)")
print(f"Outlier range: {outliers.min():.0f} ms to {outliers.max():.0f} ms")

**What to notice:** The outliers are the requests that violate user expectations the most. Even if only 5% of requests are outliers, those represent real users waiting. In industry, you would investigate: are they all hitting the same endpoint? The same server? Happening at the same time of day?

### 1.5 Grouped Summaries with `groupby`

So far we have looked at all requests together. But our data has natural groups: different endpoints, different servers, different user types. The [`groupby`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html) function lets us compute summaries *per group*, which is where things get interesting.


In [ ]:
# Summary statistics per endpoint
by_endpoint = df.groupby('endpoint')['response_time_ms'].agg(['count', 'mean', 'median', 'std'])
by_endpoint = by_endpoint.sort_values('mean', ascending=False)
print("Response time by endpoint:")
print(by_endpoint.round(2))
print()

# Summary statistics per server
by_server = df.groupby('server')['response_time_ms'].agg(['count', 'mean', 'median', 'std'])
by_server = by_server.sort_values('mean', ascending=False)
print("Response time by server:")
print(by_server.round(2))

**What to notice:** Which endpoint is the slowest on average? Is there one server that stands out? Compare the mean and median within each group — if they differ a lot, that group has more skew (more extreme outliers). Also look at the standard deviation: a high std relative to the mean suggests highly variable performance, which is bad for user experience.

### 1.6 The `describe()` function

The [`.describe()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html) function gives you a quick statistical summary. It computes count, mean, std, min, 25th/50th/75th percentiles, and max in one call. This is often the first thing you run on a new dataset to get oriented.


In [ ]:
# describe() on the full dataframe gives summaries for all numeric columns
df.describe().round(2)

---
## Section 1 Exercises

### Exercise 1.1: Error rates by endpoint

Not all requests succeed. HTTP status codes in the 400–500 range indicate errors (400 = bad request, 404 = not found, 500 = server error), while 200 and 201 indicate success.

**Task:** Create a boolean column `is_error` that is `True` when the status code is *not* 200 or 201. Then use `groupby` to compute the error count, total request count, and error rate for each endpoint. Sort by error rate descending.

**Think about:** Which endpoint has the highest error rate? Is it the same one that is slowest? A high error rate on a heavily-used endpoint is more impactful than on a rarely-used one — look at the total request count too.


In [ ]:
# Create a boolean column for errors
df['is_error'] = ~df['status_code'].isin([200, 201])

# Compute error statistics per endpoint
error_by_endpoint = df.groupby('endpoint').agg(
    error_count=('is_error', 'sum'),
    total_requests=('is_error', 'count'),
    error_rate=('is_error', 'mean')
).sort_values('error_rate', ascending=False)

error_by_endpoint['error_rate'] = error_by_endpoint['error_rate'].round(3)
print(error_by_endpoint)

### Exercise 1.2: Free vs. premium user experience

Many web services offer tiered plans where premium users get priority. Let's check whether our data reflects this.

**Task:** Use `groupby` on the `user_type` column to compute the count, mean, median, and standard deviation of response times for free and premium users.

**Think about:** Is there a meaningful difference between the two groups? How would you decide whether a difference of, say, 20 ms is "meaningful" versus just noise? (We'll learn formal tools for this — hypothesis testing — later in the course.)


In [ ]:
by_user = df.groupby('user_type')['response_time_ms'].agg(['count', 'mean', 'median', 'std'])
print(by_user.round(2))

### Exercise 1.3: The p95 by endpoint

In industry, the **95th percentile (p95)** response time is one of the most closely watched metrics. It answers: "What is the worst experience for 95% of users?" An SLA might guarantee p95 < 500 ms.

**Task:** Compute the 95th percentile of response time for each endpoint using `.quantile(0.95)`. Sort descending.

**Think about:** Which endpoints would violate a 500 ms SLA? How does this compare to the mean you computed earlier? The p95 is often 2–5× the mean for skewed data — verify whether that is true here.


In [ ]:
p95_by_endpoint = df.groupby('endpoint')['response_time_ms'].quantile(0.95).sort_values(ascending=False)
print("95th percentile response time by endpoint:")
print(p95_by_endpoint.round(2))

---
# Section 2: Histograms and Distributions

Numbers summarize; pictures reveal. A histogram shows you the **shape** of the data — where values concentrate, how spread out they are, whether the distribution is symmetric or skewed, and whether there are multiple modes or gaps.

In lecture, you saw histograms and discussed how the choice of bin width affects the story the histogram tells. Let's explore that.

### 2.1 A basic histogram

We use [`plt.hist`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hist.html) to create a histogram. We overlay vertical lines for the mean and median so we can see where the "center" falls relative to the bulk of the data.


In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(df['response_time_ms'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
plt.axvline(mean_rt, color='red', linestyle='--', linewidth=2, label=f'Mean ({mean_rt:.0f} ms)')
plt.axvline(median_rt, color='green', linestyle='--', linewidth=2, label=f'Median ({median_rt:.0f} ms)')
plt.xlabel('Response Time (ms)')
plt.ylabel('Frequency')
plt.title('Distribution of Response Times')
plt.legend()
plt.tight_layout()
plt.show()

**What to notice:** The mean line is to the right of the median line — this confirms the right skew we suspected from the numbers. The bulk of requests are fast, but a tail of slow requests pulls the mean upward. This is why the median is a better measure of the "typical" response time for skewed data like this.

### 2.2 The effect of bin width

The number of bins in a histogram controls how much detail you see. Too few bins and you lose structure; too many and you get noise. Let's see the same data with 10, 30, and 60 bins side by side.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, n_bins in zip(axes, [10, 30, 60]):
    ax.hist(df['response_time_ms'], bins=n_bins, color='steelblue', edgecolor='black', alpha=0.7)
    ax.set_xlabel('Response Time (ms)')
    ax.set_ylabel('Frequency')
    ax.set_title(f'{n_bins} bins')

plt.suptitle('Effect of Bin Count on Histogram Shape', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**What to notice:** With 10 bins, the shape is very coarse — you can see the general skew but not much detail. With 60 bins, individual bars become noisy and hard to interpret. 30 bins is a reasonable middle ground for this dataset. There is no single "right" number of bins, but a common rule of thumb is √n (here √1000 ≈ 32).

### 2.3 Custom binning with `pd.cut`

Sometimes you want to categorize a continuous variable into meaningful groups — not arbitrary equal-width bins, but categories that make domain sense. The [`pd.cut`](https://pandas.pydata.org/docs/reference/api/pandas.cut.html) function does exactly this.

For response times, we might define performance categories that an operations team would use:


In [ ]:
# Define meaningful performance categories
bins = [0, 200, 400, 600, 10000]
labels = ['Fast (<200ms)', 'Normal (200-400ms)', 'Slow (400-600ms)', 'Very Slow (>600ms)']

df['response_category'] = pd.cut(df['response_time_ms'], bins=bins, labels=labels, right=False)

counts = df['response_category'].value_counts().sort_index()
print("Performance categories:")
for cat, count in counts.items():
    print(f"  {cat:<25s}: {count:4d} requests ({100*count/len(df):.1f}%)")

**What to notice:** What fraction of requests fall into the "Slow" or "Very Slow" categories? Even a small percentage can represent a lot of unhappy users if your app handles millions of requests per day.

### 2.4 KDE plots: a smooth alternative to histograms

A **Kernel Density Estimate (KDE)** is a smooth curve that estimates the underlying probability density function. It avoids the binning artifacts of histograms and makes it easy to overlay multiple distributions for comparison.

We can overlay a KDE on top of a histogram (using `density=True` to normalize the histogram to the same scale as the KDE).


In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(df['response_time_ms'], bins=30, density=True, alpha=0.4, color='steelblue', label='Histogram')
sns.kdeplot(data=df, x='response_time_ms', color='red', linewidth=2, label='KDE')
plt.xlabel('Response Time (ms)')
plt.ylabel('Density')
plt.title('Response Time Distribution: Histogram + KDE')
plt.legend()
plt.tight_layout()
plt.show()

**What to notice:** The KDE smooths over the binning artifacts and gives you a clearer sense of the overall shape. Notice how it confirms the right skew: a peak on the left and a long tail to the right.

### 2.5 Comparing distributions across groups

One of the most powerful uses of histograms is comparing groups. Here we create a separate histogram for each endpoint so we can see whether some are faster or slower.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, endpoint in enumerate(sorted(df['endpoint'].unique())):
    data = df[df['endpoint'] == endpoint]['response_time_ms']
    axes[i].hist(data, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
    axes[i].axvline(data.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {data.mean():.0f}')
    axes[i].axvline(data.median(), color='green', linestyle='--', linewidth=2, label=f'Median: {data.median():.0f}')
    axes[i].set_title(endpoint, fontsize=11)
    axes[i].legend(fontsize=8)
    axes[i].set_xlabel('Response Time (ms)')

# Hide the unused 6th panel
axes[-1].axis('off')
fig.suptitle('Response Time Distribution by Endpoint', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**What to notice:** Do all endpoints have the same shape, or are some more skewed than others? Is the search endpoint noticeably slower (as we might expect — search queries involve more computation)? Also compare the spread: a wide histogram means inconsistent performance, which is often worse than being consistently slow.


---
## Section 2 Exercises

### Exercise 2.1: Distribution by server

We looked at distributions by endpoint. Now let's examine whether the **server** handling the request makes a difference.

**Task:** Create a figure with one histogram per server (side by side). Add vertical lines for the mean. Use the same number of bins for all three so they are visually comparable.

**Think about:** Does one server have a noticeably different distribution? A heavier tail? A higher mean? If you were an operations engineer, which server would you investigate first?


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, server in zip(axes, sorted(df['server'].unique())):
    data = df[df['server'] == server]['response_time_ms']
    ax.hist(data, bins=25, color='steelblue', edgecolor='black', alpha=0.7)
    ax.axvline(data.mean(), color='red', linestyle='--', linewidth=2,
               label=f'Mean: {data.mean():.0f} ms')
    ax.set_title(f'{server} (n={len(data)})', fontsize=11)
    ax.set_xlabel('Response Time (ms)')
    ax.legend(fontsize=9)

fig.suptitle('Response Time Distribution by Server', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Exercise 2.2: KDE comparison of free vs. premium users

Earlier we computed summary statistics for free vs. premium users. A KDE overlay lets us see the *full distributional difference*, not just a couple of numbers.

**Task:** Use `sns.kdeplot` with `hue='user_type'` to overlay KDE curves for free and premium users on the same plot. Use `fill=True` and `common_norm=False` (so each curve is normalized independently).

**Think about:** Where do the two curves diverge most? Is the difference mainly in the center of the distribution, or in the tails? A small shift in the median might not matter much, but if premium users have far fewer extreme outliers, that is a real quality-of-service advantage.


In [ ]:
plt.figure(figsize=(10, 6))
sns.kdeplot(data=df, x='response_time_ms', hue='user_type', fill=True,
            common_norm=False, alpha=0.5)
plt.xlabel('Response Time (ms)')
plt.ylabel('Density')
plt.title('Response Time Distribution: Free vs Premium Users')
plt.tight_layout()
plt.show()

---
# Section 3: Categorical Data and Bar Plots

Not all data is numerical. Our dataset has several **categorical variables**: endpoint, server, user type, HTTP method, and status code. For categorical data, the appropriate summaries are **counts** and **proportions**, and the appropriate visualizations are **bar plots** and **heatmaps**.

### 3.1 Counting with `value_counts` and simple bar plots

The [`value_counts()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html) function counts how many times each category appears. A bar plot turns those counts into a visual comparison.


In [ ]:
# How many requests hit each endpoint?
endpoint_counts = df['endpoint'].value_counts()

plt.figure(figsize=(10, 5))
endpoint_counts.plot(kind='bar', color='steelblue', edgecolor='black', alpha=0.7)
plt.xlabel('Endpoint')
plt.ylabel('Number of Requests')
plt.title('Request Volume by Endpoint')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**What to notice:** Are requests evenly distributed across endpoints, or do some get far more traffic? The busiest endpoint is the one where performance improvements would benefit the most users.

### 3.2 Status code distribution

Let's visualize the distribution of HTTP status codes. We color-code them: green for success (200, 201), red for errors.


In [ ]:
status_counts = df['status_code'].value_counts().sort_index()
colors = ['green' if code in [200, 201] else 'red' for code in status_counts.index]

plt.figure(figsize=(8, 5))
status_counts.plot(kind='bar', color=colors, alpha=0.7, edgecolor='black')
plt.xlabel('Status Code')
plt.ylabel('Count')
plt.title('HTTP Status Code Distribution')
plt.tight_layout()
plt.show()

**What to notice:** The vast majority of requests succeed (200/201). But even a small number of 500 errors is concerning — those represent server crashes. The 404s may indicate broken links or misconfigured clients.

### 3.3 Cross-tabulations and grouped bar plots

A **cross-tabulation** (using [`pd.crosstab`](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html)) counts occurrences for every combination of two categorical variables. This is the categorical equivalent of a grouped summary. We can visualize it as a grouped bar plot.


In [ ]:
ct = pd.crosstab(df['endpoint'], df['status_code'])

plt.figure(figsize=(12, 6))
ct.plot(kind='bar', width=0.8, edgecolor='black', alpha=0.7)
plt.xlabel('Endpoint')
plt.ylabel('Count')
plt.title('HTTP Status Codes by Endpoint')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Status Code')
plt.tight_layout()
plt.show()

### 3.4 Heatmaps for two-way tables

When a cross-tabulation has many cells, a **heatmap** can be easier to read than a grouped bar plot. Color intensity encodes the count, so you can spot patterns at a glance.


In [ ]:
ct_endpoint_server = pd.crosstab(df['endpoint'], df['server'])

plt.figure(figsize=(8, 5))
sns.heatmap(ct_endpoint_server, annot=True, fmt='d', cmap='YlOrRd',
            cbar_kws={'label': 'Request Count'})
plt.xlabel('Server')
plt.ylabel('Endpoint')
plt.title('Request Volume: Endpoint × Server')
plt.tight_layout()
plt.show()

**What to notice:** Is the traffic evenly distributed across servers, or is one server handling a disproportionate share of one endpoint? An imbalanced load could explain why that server is slower.


---
## Section 3 Exercises

### Exercise 3.1: User type × server heatmap

**Task:** Create a cross-tabulation of `user_type` vs. `server` and visualize it as a heatmap.

**Think about:** Are premium users distributed evenly across servers, or are they routed preferentially to certain machines? If premium users are concentrated on a faster server, that could explain the performance difference we saw earlier.


In [ ]:
ct_user_server = pd.crosstab(df['user_type'], df['server'])

plt.figure(figsize=(8, 4))
sns.heatmap(ct_user_server, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Server')
plt.ylabel('User Type')
plt.title('Request Volume: User Type × Server')
plt.tight_layout()
plt.show()

### Exercise 3.2: HTTP method distribution by endpoint

Different endpoints may handle different types of requests. A GET request retrieves data; a POST request submits data (and may be slower because it involves writes).

**Task:** Create a cross-tabulation of `endpoint` vs. `method` and visualize it as a grouped bar plot.

**Think about:** Do some endpoints only handle GETs while others handle both? Does the method mix explain some of the performance differences you observed earlier?


In [ ]:
method_by_endpoint = pd.crosstab(df['endpoint'], df['method'])

plt.figure(figsize=(12, 6))
method_by_endpoint.plot(kind='bar', width=0.8, edgecolor='black', alpha=0.7)
plt.xlabel('Endpoint')
plt.ylabel('Count')
plt.title('HTTP Method Distribution by Endpoint')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Method')
plt.tight_layout()
plt.show()

---
# Section 4: Multi-panel Figures and Advanced Plots

Real analysis rarely consists of a single plot. You typically need to compare several views of the data side by side. This section covers how to create **multi-panel figures** and introduces some more expressive plot types.

### 4.1 Box plots

A **box plot** displays the median, IQR (the box), and outliers in a single compact visual. It is one of the best tools for comparing distributions across groups because it encodes five summary statistics at once.

Recall from lecture: the box spans Q1 to Q3, the line inside is the median, the whiskers extend to the most extreme non-outlier points, and dots beyond the whiskers are outliers (using the 1.5 × IQR rule).


In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='endpoint', y='response_time_ms', palette='Set2')
plt.xlabel('Endpoint')
plt.ylabel('Response Time (ms)')
plt.title('Response Time by Endpoint (Box Plot)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**What to notice:** Compare the box heights (IQR) — a tall box means more variability. Compare the whisker lengths — long upper whiskers mean a heavy right tail. The individual dots above the whiskers are the outliers we detected numerically earlier.

### 4.2 Combining box plots and strip plots

Box plots summarize but hide the individual data points. A **strip plot** shows every data point. Overlaying both gives you the best of both worlds: the summary structure of the box plot with the granularity of raw data.


In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='endpoint', y='response_time_ms', color='white', width=0.6)
sns.stripplot(data=df, x='endpoint', y='response_time_ms', color='steelblue',
              alpha=0.3, size=3, jitter=True)
plt.xlabel('Endpoint')
plt.ylabel('Response Time (ms)')
plt.title('Response Time by Endpoint: Box + Strip Plot')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**What to notice:** The strip plot reveals the *density* of points at each level. You can see where the data concentrates and how the outliers relate to the bulk. This is much more informative than the box plot alone.

### 4.3 Multi-panel layouts with `plt.subplots`

The function [`plt.subplots(nrows, ncols)`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html) creates a grid of panels. You get back a figure and an array of axes objects; each axis is one panel that you can draw on independently.

Here we create a 2×2 dashboard that gives an operations team a comprehensive view of app performance at a glance.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: Overall distribution
axes[0, 0].hist(df['response_time_ms'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(mean_rt, color='red', linestyle='--', label=f'Mean: {mean_rt:.0f} ms')
axes[0, 0].set_xlabel('Response Time (ms)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Overall Distribution')
axes[0, 0].legend()

# Panel 2: By endpoint
sns.boxplot(data=df, x='endpoint', y='response_time_ms', ax=axes[0, 1], palette='Set2')
axes[0, 1].set_title('Response Time by Endpoint')
axes[0, 1].tick_params(axis='x', rotation=45)

# Panel 3: Request volume
df['endpoint'].value_counts().plot(kind='bar', ax=axes[1, 0], color='steelblue',
                                    alpha=0.7, edgecolor='black')
axes[1, 0].set_xlabel('Endpoint')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('Request Volume by Endpoint')
axes[1, 0].tick_params(axis='x', rotation=45)

# Panel 4: Status heatmap
ct = pd.crosstab(df['endpoint'], df['status_code'])
sns.heatmap(ct, annot=True, fmt='d', cmap='RdYlGn', ax=axes[1, 1])
axes[1, 1].set_title('Status Codes by Endpoint')

fig.suptitle('App Performance Dashboard', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

**What to notice:** A multi-panel figure tells a coherent story. The top-left gives the overall picture; the top-right breaks it down by endpoint; the bottom-left shows traffic volume (context for how important each endpoint is); and the bottom-right reveals error patterns. Together, they let a reader quickly assess the health of the system.

### 4.4 Scatter plots with color and size encoding

A **scatter plot** shows the relationship between two continuous variables. We can encode additional information using color and point size, creating a rich multidimensional visualization.

Here we examine whether larger payloads lead to slower response times (which would make physical sense — more data to transmit).


In [ ]:
plt.figure(figsize=(10, 6))
scatter = plt.scatter(df['payload_size_kb'], df['response_time_ms'],
                      c=df['status_code'], s=40, alpha=0.4, cmap='RdYlGn_r')
plt.xlabel('Payload Size (KB)')
plt.ylabel('Response Time (ms)')
plt.title('Response Time vs Payload Size (color = status code)')
plt.colorbar(scatter, label='Status Code')
plt.tight_layout()
plt.show()

**What to notice:** Is there a visible trend (larger payloads → slower responses)? Are the error responses (red/dark points, status 400/500) concentrated in any particular region? If errors cluster at large payloads, it might indicate a size limit being hit. If they are scattered randomly, the errors are likely unrelated to payload size.


---
## Section 4 Exercises

### Exercise 4.1: Build a performance dashboard

You've seen individual plots throughout this tutorial. Now combine them into a **2×3 dashboard** that gives a complete overview of your application's health.

**Task:** Create a 2×3 figure with the following panels:
1. Histogram of response times
2. Box plot by endpoint
3. Box plot by server
4. Bar chart of request volume by endpoint
5. Bar chart of status code distribution
6. Bar chart of error rate by endpoint

**Think about:** If you had to present this to a non-technical manager, which panels convey the most important information? A good dashboard tells a story: "here is the overall picture, here are the problem areas, and here is how significant they are."


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Panel 1: Overall histogram
axes[0, 0].hist(df['response_time_ms'], bins=25, color='steelblue', alpha=0.7, edgecolor='black')
axes[0, 0].axvline(mean_rt, color='red', linestyle='--', label=f'Mean: {mean_rt:.0f}')
axes[0, 0].set_title('Response Time Distribution')
axes[0, 0].set_xlabel('Response Time (ms)')
axes[0, 0].legend()

# Panel 2: Box plot by endpoint
sns.boxplot(data=df, x='endpoint', y='response_time_ms', ax=axes[0, 1], palette='Set2')
axes[0, 1].set_title('By Endpoint')
axes[0, 1].tick_params(axis='x', rotation=45)

# Panel 3: Box plot by server
sns.boxplot(data=df, x='server', y='response_time_ms', ax=axes[0, 2], palette='Set1')
axes[0, 2].set_title('By Server')

# Panel 4: Request volume
df['endpoint'].value_counts().plot(kind='bar', ax=axes[1, 0], color='steelblue',
                                    alpha=0.7, edgecolor='black')
axes[1, 0].set_title('Request Volume')
axes[1, 0].tick_params(axis='x', rotation=45)

# Panel 5: Status codes
df['status_code'].value_counts().sort_index().plot(kind='bar', ax=axes[1, 1], alpha=0.7, edgecolor='black')
axes[1, 1].set_title('Status Code Distribution')

# Panel 6: Error rate by endpoint
error_rate = (df.groupby('endpoint')['is_error'].mean() * 100).sort_values(ascending=False)
error_rate.plot(kind='bar', ax=axes[1, 2], color='indianred', alpha=0.7, edgecolor='black')
axes[1, 2].set_title('Error Rate (%) by Endpoint')
axes[1, 2].tick_params(axis='x', rotation=45)

fig.suptitle('App Performance Dashboard', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### Exercise 4.2: Time-based trends

Performance often varies by time of day — peak hours bring more traffic and potentially slower responses. Understanding these patterns helps operations teams plan capacity.

**Task:** 
1. Convert the `timestamp` column to a datetime type and extract the hour of day.
2. Group by hour and compute the mean response time, the standard deviation, and the request count.
3. Create a 2-panel figure: the top panel shows mean response time by hour (as a line plot with a ±1σ shaded band), and the bottom shows request volume by hour (as a bar chart).

**Think about:** Are the slowest hours also the busiest hours? If so, the app may be struggling under load. If the slowest hours are *not* the busiest, something else may be going on (e.g., scheduled batch jobs competing for resources).


In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['timestamp'].dt.hour

hourly = df.groupby('hour')['response_time_ms'].agg(['mean', 'count', 'std'])

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Top: Mean response time with ±1σ band
axes[0].plot(hourly.index, hourly['mean'], marker='o', linewidth=2, color='steelblue')
axes[0].fill_between(hourly.index,
                      hourly['mean'] - hourly['std'],
                      hourly['mean'] + hourly['std'],
                      alpha=0.2, color='steelblue')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Mean Response Time (ms)')
axes[0].set_title('Average Response Time by Hour of Day (shaded = ± 1 std dev)')
axes[0].grid(alpha=0.3)

# Bottom: Request volume
axes[1].bar(hourly.index, hourly['count'], color='steelblue', alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Request Count')
axes[1].set_title('Request Volume by Hour of Day')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

### Exercise 4.3: Violin plots for multi-dimensional comparison

A **violin plot** is like a box plot but shows the full density shape on each side, giving you a richer view of the distribution.

**Task:** Create a 2×2 figure:
1. Top-left: violin plot of response time by user type
2. Top-right: bar chart of error rate by user type
3. Bottom-left: violin plot of response time by server
4. Bottom-right: bar chart of error rate by server

**Think about:** The violin shape tells you more than the box plot — you can see whether the distribution is unimodal or bimodal, symmetric or skewed. Combined with the error rates, you get a full picture of both speed and reliability per group.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top-left: Violin by user type
sns.violinplot(data=df, x='user_type', y='response_time_ms', ax=axes[0, 0], palette='pastel')
axes[0, 0].set_title('Response Time by User Type')

# Top-right: Error rate by user type
error_by_user = (df.groupby('user_type')['is_error'].mean() * 100)
error_by_user.plot(kind='bar', ax=axes[0, 1], color=['steelblue', 'orange'],
                   alpha=0.7, edgecolor='black')
axes[0, 1].set_title('Error Rate (%) by User Type')
axes[0, 1].set_ylabel('Error Rate (%)')
axes[0, 1].tick_params(axis='x', rotation=0)

# Bottom-left: Violin by server
sns.violinplot(data=df, x='server', y='response_time_ms', ax=axes[1, 0], palette='Set1')
axes[1, 0].set_title('Response Time by Server')

# Bottom-right: Error rate by server
error_by_server = (df.groupby('server')['is_error'].mean() * 100)
error_by_server.plot(kind='bar', ax=axes[1, 1], color='steelblue', alpha=0.7, edgecolor='black')
axes[1, 1].set_title('Error Rate (%) by Server')
axes[1, 1].set_ylabel('Error Rate (%)')
axes[1, 1].tick_params(axis='x', rotation=0)

fig.suptitle('Performance Comparison: User Type and Server', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
# Summary: What you learned

In this tutorial you practiced the core data analysis workflow: **summarize → visualize → compare → investigate**.

**Section 1 — Data Summaries:** You computed mean, median, percentiles, standard deviation, and IQR. You learned that the mean and median can disagree when data is skewed, that percentiles (especially p95/p99) are critical for understanding worst-case performance, and that `groupby` lets you compare groups.

**Section 2 — Histograms and Distributions:** You saw how bin width affects the histogram's story, how KDE provides a smooth alternative, and how faceted histograms reveal differences across groups.

**Section 3 — Categorical Data:** You used `value_counts`, bar plots, cross-tabulations, and heatmaps to summarize and compare categorical variables.

**Section 4 — Advanced Visualization:** You built box plots, combined box + strip plots, created multi-panel dashboards, and explored scatter plots with color encoding.

### Connection to your group experiment

When you collect data from your group experiment later in the course, you will use exactly these tools:
- **Summary statistics** to describe your measurements
- **Histograms and KDE** to visualize the distribution of your outcome variable
- **Grouped summaries and box plots** to compare your experimental conditions
- **Multi-panel figures** to present your results in your report

The techniques in this tutorial are the foundation of the data analysis section of your group report.
